# Axolotl telencephalon — CellSTIC tutorial

**Dataset:** Spatial telencephalon; per-batch under data/axolotl_telencephalon/<batch>/; split from combined raw/development.h5ad / regeneration.h5ad. Ligand-receptor map data/axolotl_telencephalon/LR.csv.


## Step 1 — Setup


In [ ]:
import sys
import base64
from pathlib import Path
from typing import Dict

%load_ext autoreload
%autoreload 2

_cwd = Path.cwd()
project_root = _cwd if (_cwd / "model").is_dir() else _cwd.parent
sys.path.insert(0, str(project_root))

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sparse
import torch
from IPython.display import HTML, SVG, display
from scipy.spatial.distance import pdist, squareform

from model.train import CellSTICConfig
from pipeline import run_cellstic
from pipeline.analyzer import TimeSequenceAnalysis
from utils.data import SpatialPreprocessorUtils
from utils.metrics import MetricsComputer, SpatialVisualizer
from utils.tools.seed_utils import set_global_seed

set_global_seed()

def load_ligand_receptor_map_with_edge_types(file_path):
    """Load ligand/receptor CSV without depending on the loader package."""
    lr_df = pd.read_csv(file_path)

    ligand_col = receptor_col = idx_col = None
    for col in lr_df.columns:
        lower = str(col).lower()
        if "ligand" in lower and ligand_col is None:
            ligand_col = col
        elif "receptor" in lower and receptor_col is None:
            receptor_col = col
        elif "idx" in lower and idx_col is None:
            idx_col = col

    if ligand_col is None or receptor_col is None:
        raise ValueError(
            f"CSV {file_path} must contain ligand and receptor columns; "
            f"found {list(lr_df.columns)}"
        )

    ligand_receptor_map = {}
    edge_type_map = {}
    for row_idx, row in lr_df.iterrows():
        ligand = str(row[ligand_col]).strip()
        receptor = str(row[receptor_col]).strip()
        if not ligand or not receptor or ligand.lower() == "nan" or receptor.lower() == "nan":
            continue
        ligand_receptor_map.setdefault(ligand, [])
        if receptor not in ligand_receptor_map[ligand]:
            ligand_receptor_map[ligand].append(receptor)
        idx = int(row[idx_col]) - 1 if idx_col is not None else len(edge_type_map)
        edge_type_map[f"{ligand}:{receptor}"] = idx
    return ligand_receptor_map, edge_type_map


def display_svg_grid(paths, n_cols=None, max_width=280):
    paths = [Path(p) for p in paths]
    if not paths:
        print("No SVG files found to display.")
        return
    if n_cols is None:
        n_cols = min(3, len(paths))
    rows = []
    for i in range(0, len(paths), n_cols):
        cells = []
        for path in paths[i : i + n_cols]:
            uri = "data:image/svg+xml;base64," + base64.b64encode(path.read_bytes()).decode("ascii")
            cells.append(
                f'<td style="width:{100 / n_cols:.2f}%;padding:4px;text-align:center">'
                f'<img src="{uri}" style="width:100%;max-width:{max_width}px;height:auto;"/>'
                f"</td>"
            )
        cells.extend(f'<td style="width:{100 / n_cols:.2f}%"></td>' for _ in range(n_cols - len(cells)))
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f'<table style="width:100%;table-layout:fixed;border-collapse:collapse"><tbody>{"".join(rows)}</tbody></table>'))


## Step 2 — Configuration


In [ ]:
cfg = CellSTICConfig()
cfg.model.graph.cluster_top_k = 10
cfg.model.graph.cluster_size = 30
cfg.model.graph.expression_percentile = 95
cfg.model.graph.n_spots = 4
cfg.model.tree.hierarchy_method = "balanced"
cfg.train.ccc.epochs = 200
cfg.train.feat.epochs = 1000
cfg.train.feat.learning_rate = 0.003
cfg.train.feat.n_clusters = 10
cfg.train.feat.weight_modality = 0.4
cfg.train.feat.entropy_weight = 0.4

RUN_SPLIT_DATA = False
RUN_SINGLE_BATCH_PIPELINE = False
RUN_SINGLE_BATCH_VIS = False
RUN_TIME_SERIES_ANALYSIS = True
USE_CACHE = True

BATCH_NAME = "develop_44"
COLOR_KEY = "Annotation"
TIME_MODE = "development"  # choices: "development", "regeneration"

dataset_root = project_root / "data" / "axolotl_telencephalon"
work_dir = dataset_root / BATCH_NAME
raw_path = work_dir / "raw"
preprocess_path = work_dir / "preprocess"
model_path = work_dir / "model"
result_path = work_dir / "result"
analysis_path = work_dir / "analysis"
lr_path = dataset_root / "LR.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for path in [raw_path, preprocess_path, model_path, result_path, analysis_path]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Dataset root:     {dataset_root}")
print(f"Batch:            {BATCH_NAME}")
print(f"Device:           {device}")
print(f"Result path:      {result_path}")
print(f"Analysis path:    {analysis_path}")
print(f"LR path:          {lr_path}")


## Step 3 — Optional Data Splitting

Use this only when starting from combined raw files under data/axolotl_telencephalon/raw/.


In [ ]:
_BATCH_TO_DIR_DEVELOPMENT: Dict[str, str] = {
    "Batch1_Adult_telencephalon_rep2_DP8400015234BLA3_1": "develop_Adult",
    "Injury_control_FP200000239BL_E3": "develop_Juvenile",
    "Meta_telencephalon_rep1_DP8400015234BLB2_1": "develop_Metamorphosed",
    "Stage44_telencephalon_rep2_FP200000239BL_E4": "develop_44",
    "Stage54_telencephalon_rep2_DP8400015649BRD6_2": "develop_54",
    "Stage57_telencephalon_rep2_DP8400015649BRD5_1": "develop_57",
}

_BATCH_TO_DIR_REGENERATION: Dict[str, str] = {
    "Injury_2DPI_rep1_SS200000147BL_D5": "regeneration_2",
    "Injury_5DPI_rep1_SS200000147BL_D2": "regeneration_5",
    "Injury_10DPI_rep1_SS200000147BL_B5": "regeneration_10",
    "Injury_15DPI_rep4_FP200000266TR_E4": "regeneration_15",
    "Injury_20DPI_rep2_SS200000147BL_B4": "regeneration_20",
    "Injury_30DPI_rep2_FP200000264BL_A6": "regeneration_30",
    "Injury_60DPI_rep3_FP200000264BL_A6": "regeneration_60",
    "Injury_control_FP200000239BL_E3": "regeneration_control",
}

_SPLIT_MAP_BY_H5AD_NAME: Dict[str, Dict[str, str]] = {
    "development.h5ad": _BATCH_TO_DIR_DEVELOPMENT,
    "regeneration.h5ad": _BATCH_TO_DIR_REGENERATION,
}

if RUN_SPLIT_DATA:
    src_dir = dataset_root / "raw"
    h5ad_files = sorted(src_dir.glob("*.h5ad"))
    if not h5ad_files:
        raise FileNotFoundError(f"No .h5ad files found under {src_dir}")

    for h5ad_path in h5ad_files:
        batch_map = _SPLIT_MAP_BY_H5AD_NAME.get(h5ad_path.name.lower())
        if batch_map is None:
            raise ValueError(f"{h5ad_path.name}: no split mapping defined.")

        adata = ad.read_h5ad(h5ad_path, backed="r")
        if "Batch" not in adata.obs.columns:
            raise KeyError(f"{h5ad_path.name}: obs does not contain 'Batch' column")

        batch_series = adata.obs["Batch"].astype(str)
        batch_folders = batch_series.map(lambda batch: batch_map[batch])

        for folder in sorted(batch_folders.unique()):
            mask = (batch_folders == folder).to_numpy()
            out_dir = dataset_root / folder / "raw"
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / h5ad_path.name
            adata[mask].to_memory().write_h5ad(out_path)
            print(f"Saved: {out_path}")
else:
    print("Skip data splitting.")


## Step 4 — Load and Preprocess

Normalize pipe-formatted gene names, ensure spatial coordinates, build features and spatial distances, and cache the preprocessed batch AnnData.


In [ ]:
_CACHE_NAME = "preprocessed_RNA.h5ad"


def normalize_var_names_pipe_format(adata):
    if adata is None or adata.n_vars == 0:
        return
    names = pd.Index(adata.var_names.astype(str))
    if not names.str.contains(r"\|").any():
        return
    new_names = names.str.split("|", n=1, expand=False).str[0].str.strip()
    adata.var_names = pd.Index(new_names).astype(str)
    adata.var_names_make_unique()


def load_axolotl_telencephalon_inline(raw_path, preprocess_path, use_cache=True, lr_path=None):
    raw_path = Path(raw_path)
    preprocess_path = Path(preprocess_path)

    lr_map = None
    if lr_path is not None and Path(lr_path).exists():
        lr_map, _ = load_ligand_receptor_map_with_edge_types(lr_path)

    cache_path = preprocess_path / _CACHE_NAME
    if use_cache and cache_path.exists():
        rna_adata = sc.read_h5ad(cache_path)
        normalize_var_names_pipe_format(rna_adata)
        return rna_adata, lr_map

    raw_files = [path for path in raw_path.glob("*.h5ad") if path.name != _CACHE_NAME]
    if not raw_files:
        raise FileNotFoundError(f"No raw h5ad found under {raw_path.absolute()}")
    rna_adata = sc.read_h5ad(raw_files[0])
    normalize_var_names_pipe_format(rna_adata)

    if "spatial" not in rna_adata.obsm:
        if "x" in rna_adata.obs and "y" in rna_adata.obs:
            rna_adata.obsm["spatial"] = rna_adata.obs[["x", "y"]].to_numpy()
        else:
            raise ValueError("Spatial coordinates not found in obsm['spatial'] or obs['x', 'y'].")

    if "feat" in rna_adata.obsm and "spatial_distances" in rna_adata.obsp:
        if use_cache:
            preprocess_path.mkdir(parents=True, exist_ok=True)
            rna_adata.write_h5ad(cache_path)
        return rna_adata, lr_map

    rna_adata = rna_adata.copy()
    rna_adata.raw = rna_adata.copy()
    rna_adata.var["mt"] = rna_adata.var_names.astype(str).str.startswith("MT-")
    sc.pp.calculate_qc_metrics(rna_adata, percent_top=None, log1p=False, inplace=True)
    sc.pp.highly_variable_genes(rna_adata, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(rna_adata, target_sum=1e4)
    sc.pp.log1p(rna_adata)
    rna_adata.obsm["feat"] = SpatialPreprocessorUtils.pca(
        rna_adata[:, rna_adata.var["highly_variable"]], n_comps=500
    )
    rna_adata.obsp["spatial_distances"] = sparse.csr_matrix(
        squareform(pdist(rna_adata.obsm["spatial"], metric="euclidean"))
    )

    if use_cache:
        preprocess_path.mkdir(parents=True, exist_ok=True)
        rna_adata.write_h5ad(cache_path)
    return rna_adata, lr_map


if RUN_SINGLE_BATCH_PIPELINE or RUN_SINGLE_BATCH_VIS:
    rna_adata, lr_map = load_axolotl_telencephalon_inline(
        raw_path,
        preprocess_path,
        use_cache=USE_CACHE,
        lr_path=lr_path,
    )
    if lr_map is None:
        raise FileNotFoundError(f"Ligand-receptor CSV not found: {lr_path}")
    print(f"RNA shape:        {rna_adata.shape}")
    print(f"RNA feat:         {rna_adata.obsm['feat'].shape}")
    print(f"Ligand entries:   {len(lr_map)}")
else:
    print("Skip single-batch loading; enable RUN_SINGLE_BATCH_PIPELINE or RUN_SINGLE_BATCH_VIS to load this batch.")


## Step 5 — Run CellSTIC


In [ ]:
if RUN_SINGLE_BATCH_PIPELINE:
    result = run_cellstic(
        modality_datas=[rna_adata],
        ligand_receptor_map=lr_map,
        model_path=model_path,
        output_path=result_path,
        config=cfg,
        device=device,
        auto_n_clusters=7,
    )
    print(f"AnnData saved to {result.adata_path}")
    print("Single-batch pipeline finished.")
else:
    print("Skip pipeline (set RUN_SINGLE_BATCH_PIPELINE=True to run this step).")


## Step 6 — Optional Spatial + UMAP Visualization


In [ ]:
if RUN_SINGLE_BATCH_VIS:
    if COLOR_KEY not in rna_adata.obs.columns:
        raise ValueError(f"obs[{COLOR_KEY!r}] not found; cannot color region plots.")

    adata_vis = rna_adata.copy()
    adata_vis.obs["cluster"] = adata_vis.obs[COLOR_KEY].astype(str).astype("category")

    save_path_domain = analysis_path / "domain" / "spatial_domain.svg"
    save_path_domain.parent.mkdir(parents=True, exist_ok=True)
    SpatialVisualizer.generate_spatial_domain_visualization(
        adata_source=adata_vis,
        save_path=str(save_path_domain),
    )
    display(SVG(filename=str(save_path_domain)))

    MetricsComputer.run_region_umap_metrics_export(
        adata_vis,
        save_dir=analysis_path / "domain",
        feature_key="feat",
        cluster_key="cluster",
    )
    display_svg_grid(sorted((analysis_path / "domain").glob("*.svg")), n_cols=2)
    print("Visualization export finished.")
else:
    print("Skip optional spatial / UMAP export.")


## Step 7 — Optional Cross-stage Time-series Analysis


In [ ]:
def ensure_time_sequence_cci_layout(cci_root: Path, stages, organ: str = "telencephalon"):
    for stage in stages:
        stage_dir = cci_root / stage
        if not stage_dir.is_dir():
            continue
        organ_dir = stage_dir / organ
        organ_dir.mkdir(parents=True, exist_ok=True)
        for file_path in stage_dir.glob("*.csv"):
            if not file_path.is_file() or file_path.name.upper() == "LR.CSV":
                continue
            link = organ_dir / file_path.name
            if link.exists() or link.is_symlink():
                continue
            try:
                link.symlink_to(file_path.resolve())
            except OSError:
                pass


def ensure_time_sequence_spatial_layout(spatial_root: Path, stages, organ: str = "telencephalon"):
    for stage in stages:
        src = spatial_root / stage / "preprocess" / "preprocessed_RNA.h5ad"
        if not src.is_file():
            continue
        target_dir = spatial_root / stage / "preprocess" / organ
        target_dir.mkdir(parents=True, exist_ok=True)
        dst = target_dir / "preprocessed_RNA_filtered.h5ad"
        if dst.exists() or dst.is_symlink():
            continue
        try:
            dst.symlink_to(src.resolve())
        except OSError:
            pass


if TIME_MODE == "development":
    batch_list = [
        "develop_44",
        "develop_54",
        "develop_57",
        "develop_Juvenile",
        "develop_Adult",
        "develop_Metamorphosed",
    ]
    time_work_dir = project_root / "data" / "axolotl_develop_cci"
elif TIME_MODE == "regeneration":
    batch_list = [
        "regeneration_2",
        "regeneration_5",
        "regeneration_10",
        "regeneration_15",
        "regeneration_20",
        "regeneration_30",
        "regeneration_60",
        "regeneration_control",
    ]
    time_work_dir = project_root / "data" / "axolotl_regeneration_cci"
else:
    raise ValueError("TIME_MODE must be 'development' or 'regeneration'")

time_output_path = time_work_dir / "output"
time_raw_path = time_work_dir / "raw"
time_output_path.mkdir(parents=True, exist_ok=True)

print("TIME_MODE =", TIME_MODE)
print("batch_list =", batch_list)
print("time_work_dir =", time_work_dir)


## Step 8 — Run Time-series Analysis


In [ ]:
if RUN_TIME_SERIES_ANALYSIS:
    ensure_time_sequence_cci_layout(time_raw_path, batch_list, organ="telencephalon")
    ensure_time_sequence_spatial_layout(dataset_root, batch_list, organ="telencephalon")

    lr_filter = ["WNT7B-FZD5"]
    annotation_filter = ["telencephalon"]
    region_lr_map = {"telencephalon": "WNT7B-FZD5"}

    time_sequence_analysis = TimeSequenceAnalysis(
        stages=batch_list,
        raw_path=time_raw_path,
        spatial_root=str(dataset_root),
        output_path=time_output_path,
        annotation_filter=annotation_filter,
        lr_filter=lr_filter,
        fig_format="svg",
        cell_type_key="Annotation",
    )

    time_sequence_analysis.count_cell_number(recompute=False, font_size=12)
    time_sequence_analysis.count_edge_number(recompute=False, font_size=12)
    time_sequence_analysis.plot_efficiency_metrics(recompute=False, region_lr_map=region_lr_map, font_size=8)
    time_sequence_analysis.plot_celltype_strength_bars(recompute=False, region_lr_map=region_lr_map, font_size=8)
    time_sequence_analysis.plot_graph_density_over_stages(recompute=False, font_size=12)
    time_sequence_analysis.plot_strong_nodes_over_stages(recompute=False, region_lr_map=region_lr_map, font_size=8)
    time_sequence_analysis.plot_strength_vs_distance_over_stages(recompute=False, region_lr_map=region_lr_map, font_size=12)
    time_sequence_analysis.plot_ccdf_degree_strength(recompute=False, region_lr_map=region_lr_map, font_size=8)

    print(f"Time-series analysis completed for TIME_MODE={TIME_MODE!r}")
    print(f"Time-series outputs saved under: {time_output_path}")
else:
    print("Skip time-series analysis.")
